<a href="https://colab.research.google.com/github/bukharilab/BurnoutSurveillanceAI/blob/main/temporal_dyn_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pipeline Overview

This notebook processes raw discharge notes and admissions data to derive several temporal features for clinicians, ultimately calculating a Temporal Inflexibility Index (TII). The pipeline consists of the following main components:

1.  **Data Loading and Initial Preprocessing**: Raw discharge notes and admissions data are loaded. Notes are joined with admissions to link them to `PROVIDER_ID` (aliased as `CLINICIAN_ID`). Timestamps are parsed, and basic features like `WORD_COUNT`, `HOUR_FLOAT`, `DATE`, `WEEK`, `MONTH`, and `AFTER_HOURS` are extracted.

2.  **Weekly Temporal Features**: Notes are grouped by `CLINICIAN_ID` and `WEEK` to calculate weekly aggregates such as note count, average word count, earliest/latest/mean/std of documentation hours, after-hours ratio, and active days.

3.  **Topic Modeling (BERTopic) & Fragmentation**: BERTopic is used to identify latent topics within the discharge notes. Each note is assigned a `TOPIC_ID`. A `topic_switch_rate` is then calculated per clinician-month to quantify temporal fragmentation, which indicates how frequently a clinician switches between topics in their notes.

4.  **Monthly Temporal Load/Scarcity Features**: Notes are grouped by `CLINICIAN_ID` and `MONTH` to compute monthly metrics like total note count, average word count, total word count, and active days. These are used to derive `notes_per_span_day`, `notes_per_active_day`, and a `temporal_scarcity_z` score, reflecting workload and the density of documentation.

5.  **Monthly Temporal Flexibility Features**: Monthly metrics like `hour_std` (variability of documentation hours), `after_hours_ratio`, and `active_days_ratio` (spread of active days) are normalized and combined to create a `flexibility_score`, indicating a clinician's control over their work timing.

6.  **Consolidated Monthly Temporal Features & TII Calculation**: All derived monthly features (note counts, word counts, hour-based metrics, after-hours ratio, active days, topic switch rate) are gathered. Proxies for `scarcity`, `flexibility`, and `fragmentation` are defined, and their Z-scores are calculated within each clinician's data. These Z-scores are then combined to compute the `Temporal Inflexibility Index (TII)` per clinician-month.

7.  **Clinician-Level Aggregation**: The monthly TII and its component Z-scores (load, fragmentation, flexibility) are aggregated to the clinician level. This involves calculating statistics such as mean, standard deviation, last value, mean of the last 3 values, and slope of the last 3 values for each metric, providing a summarized view of each clinician's temporal work patterns.

In [ ]:
import os
import numpy as np
import pandas as pd
from datetime import datetime

import matplotlib.pyplot as plt

ROOT = "/content/drive/MyDrive/Colab Notebooks/Burnout"
paths = {
    'discharge': f'{ROOT}/discharge.csv',
    'admissions': f'{ROOT}/admissions.csv',
    'monthly': f'{ROOT}/temp dyn/temporal_monthly_with_TII.csv'
}

# Preprocessing

In [ ]:
import os
import numpy as np
import pandas as pd

# -------------------------------------------------
# FILE PATHS (you already have this)
# -------------------------------------------------
NOTES_CSV      = paths['discharge']    # raw discharge.csv
ADMISSIONS_CSV = paths['admissions']   # admissions.csv

# -------------------------------------------------
# 1) LOAD DISCHARGE NOTES
# -------------------------------------------------
print("Loading discharge notes...")
notes = pd.read_csv(NOTES_CSV, low_memory=False)
print("  → shape:", notes.shape)
print("  → columns:", list(notes.columns))

# Standardize column names we need
notes = notes.rename(columns={
    "hadm_id":   "HADM_ID",
    "text":      "TEXT",
    "charttime": "CHARTTIME",
    "storetime": "STORETIME"
})

# We require HADM_ID and TEXT, and at least one of CHARTTIME / STORETIME
base_required = {"HADM_ID", "TEXT"}
time_cols     = {"CHARTTIME", "STORETIME"} & set(notes.columns)

missing_base = base_required - set(notes.columns)
if missing_base:
    raise KeyError(f"Discharge CSV missing columns: {missing_base}")

if not time_cols:
    raise KeyError("Discharge CSV must contain either 'charttime' or 'storetime'.")

# -------------------------------------------------
# 2) LOAD ADMISSIONS (TO GET PROVIDER / CLINICIAN)
# -------------------------------------------------
print("\nLoading admissions...")
adm = pd.read_csv(
    ADMISSIONS_CSV,
    usecols=["hadm_id", "admit_provider_id"],
    low_memory=False
)
adm = adm.rename(columns={
    "hadm_id":           "HADM_ID",
    "admit_provider_id": "PROVIDER_ID"
})
adm["PROVIDER_ID"] = adm["PROVIDER_ID"].astype(str)

print("  → admissions shape:", adm.shape)
print("  → admissions columns:", list(adm.columns))

# -------------------------------------------------
# 3) MERGE DISCHARGE + ADMISSIONS
# -------------------------------------------------
print("\nMerging discharge notes with admissions on HADM_ID...")
notes = notes.merge(adm, on="HADM_ID", how="inner")
notes["PROVIDER_ID"]  = notes["PROVIDER_ID"].astype(str)
notes["CLINICIAN_ID"] = notes["PROVIDER_ID"]   # alias

print("  → merged notes shape:", notes.shape)

# -------------------------------------------------
# 4) BUILD DATETIME (PREFER STORETIME) + SIMPLE FEATURES
# -------------------------------------------------
print("\nBuilding DATETIME + helper features...")

if "STORETIME" in notes.columns:
    # Prefer the real storage timestamp
    notes["DATETIME"] = pd.to_datetime(notes["STORETIME"], errors="coerce")
    print("  → DATETIME built from STORETIME.")
else:
    notes["DATETIME"] = pd.to_datetime(notes["CHARTTIME"], errors="coerce")
    print("  → DATETIME built from CHARTTIME (STORETIME not present).")

before = len(notes)
notes = notes.dropna(subset=["DATETIME"]).copy()
after = len(notes)
print(f"  → dropped {before - after} rows with NaT DATETIME; remaining: {after}")

# Basic text feature
notes["WORD_COUNT"] = notes["TEXT"].astype(str).str.split().str.len()

# Time-based helpers
notes["HOUR_FLOAT"] = (
    notes["DATETIME"].dt.hour + notes["DATETIME"].dt.minute / 60.0
)
notes["DATE"]  = notes["DATETIME"].dt.date
notes["MONTH"] = notes["DATETIME"].dt.to_period("M").apply(lambda r: r.start_time)
notes["WEEK"]  = notes["DATETIME"].dt.to_period("W").apply(lambda r: r.start_time)

# Simple after-hours flag (18:00–07:59) – now meaningful because we use STORETIME
notes["AFTER_HOURS"] = (
    (notes["DATETIME"].dt.hour >= 18) | (notes["DATETIME"].dt.hour < 8)
).astype(int)

print("\nFinal preprocessed notes columns (sample):")
print(
    notes[
        ["CLINICIAN_ID", "HADM_ID", "DATETIME",
         "WORD_COUNT", "HOUR_FLOAT", "AFTER_HOURS", "DATE", "WEEK", "MONTH"]
    ].head()
)

In [ ]:
weekly = (
    notes.groupby(["CLINICIAN_ID", "WEEK"])
    .agg(
        note_count        = ("TEXT", "count"),
        avg_word_count    = ("WORD_COUNT", "mean"),
        earliest_hour     = ("HOUR_FLOAT", "min"),
        latest_hour       = ("HOUR_FLOAT", "max"),
        hour_mean         = ("HOUR_FLOAT", "mean"),
        hour_std          = ("HOUR_FLOAT", "std"),
        after_hours_ratio = ("AFTER_HOURS", "mean"),
        days_active       = ("DATE", lambda x: x.nunique())
    )
    .reset_index()
)

In [ ]:
weekly

In [ ]:
len(set(notes['CLINICIAN_ID']))

In [ ]:
len([hour for hour in weekly['hour_std'] if hour > 0])

In [ ]:
len(notes)

In [ ]:
not_after = len([ratio for ratio in notes['AFTER_HOURS'] if ratio==0])
after = len([ratio for ratio in notes['AFTER_HOURS'] if ratio==1])

after_hours_percentage = after/len(notes) * 100

In [ ]:

import seaborn as sns
corr = weekly.drop(columns=["CLINICIAN_ID", "WEEK"]).corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Temporal Features")
plt.show()

In [ ]:
pivot = notes.pivot_table(
    index="CLINICIAN_ID",
    columns=notes["DATETIME"].dt.hour,
    values="note_id",
    aggfunc="count",
    fill_value=0
)

plt.figure(figsize=(14,10))
sns.heatmap(pivot, cmap="magma")
plt.title("Clinician Hour-of-Day Writing Heatmap")
plt.ylabel("Clinician")
plt.xlabel("Hour of Day")
plt.show()

# would be useful to do this after we figure out top 10-20 highest TII clinicians

# Semantic Fragementation with topic switching
- Need to detect how frequently topic switches accross notes from same physician
- higher switch rate == higher chanve of burnout

In [ ]:
!pip install bertopic

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

docs = notes["TEXT"].astype(str).tolist()

# Fast, GPU-friendly model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

topic_model = BERTopic(
    embedding_model=embedder,
    min_topic_size=100,          # reduce if dataset small
    verbose=True
)

topics, probs = topic_model.fit_transform(docs)

# Attach to notes
notes["TOPIC_ID"] = topics

In [ ]:
def topic_switch_rate(topic_list):
    """
    topic_list: list or array-like of topic IDs in chronological order.
    Returns the fraction of adjacent notes that switch topic.
    """
    if len(topic_list) <= 1:
        return 0.0

    # Compare adjacent values
    switches = sum(topic_list[i] != topic_list[i-1] for i in range(1, len(topic_list)))
    return switches / (len(topic_list) - 1)

In [ ]:
switch_df = (
    notes.sort_values("DATETIME")
         .groupby(["CLINICIAN_ID", "MONTH"])["TOPIC_ID"]
         .apply(lambda x: topic_switch_rate(list(x)))
         .reset_index(name="topic_switch_rate")
)

# Narrative scarcity

In [ ]:

group_cols = ["CLINICIAN_ID", "MONTH"]

monthly = (
    notes
    .groupby(group_cols)
    .agg(
        note_count       = ("TEXT", "count"),
        avg_word_count   = ("WORD_COUNT", "mean"),
        total_word_count = ("WORD_COUNT", "sum"),
        first_date       = ("DATE", "min"),
        last_date        = ("DATE", "max"),
        active_days      = ("DATE", lambda d: d.nunique())
    )
    .reset_index()
)

# span of time from first to last documented discharge in that month
monthly["span_days"] = (
    pd.to_datetime(monthly["last_date"]) -
    pd.to_datetime(monthly["first_date"])
).dt.days + 1

# avoid zero / negative spans
monthly["span_days"] = monthly["span_days"].clip(lower=1)

# docs per day across the entire span
monthly["notes_per_span_day"] = monthly["note_count"] / monthly["span_days"]

# docs per *active* documentation day
monthly["notes_per_active_day"] = monthly["note_count"] / monthly["active_days"].clip(lower=1)

monthly.head()

In [ ]:
def z_by_clin(df, col):
    return df.groupby("CLINICIAN_ID")[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-6)
    )

# z-scores within clinician
monthly["z_notes_per_active_day"] = z_by_clin(monthly, "notes_per_active_day")
monthly["z_avg_word_count"]       = z_by_clin(monthly, "avg_word_count")

# invert length: short notes → high scarcity
monthly["z_short_notes"] = -monthly["z_avg_word_count"]

# simple average → Temporal Scarcity Index
monthly["temporal_scarcity_z"] = (
    monthly["z_notes_per_active_day"] + monthly["z_short_notes"]
) / 2

monthly[[
    "CLINICIAN_ID", "MONTH",
    "note_count", "avg_word_count",
    "notes_per_active_day",
    "temporal_scarcity_z"
]].head()

In [ ]:
len([s for s in monthly['temporal_scarcity_z'] if s < -0.5])

In [ ]:
plt.figure(figsize=(6,4))
plt.hist(monthly["temporal_scarcity_z"].dropna(), bins=50)
plt.axvline(0, color="k", linestyle="--")
plt.xlabel("Temporal Scarcity (z within clinician)")
plt.ylabel("Number of clinician-months")
plt.title("Distribution of Temporal Scarcity")
plt.tight_layout()
plt.show()

# Temporal Flexibility
How much control the clinician has over the timing of work; the degree to which work can be distributed flexibly accross hours and days

3 Components:
- Variability of documentation hours (hours_std)
- After hours ratio (after_hours_ratio)
- Spread of active days per month (active_days_ratio)

High temporal flexibility:
- notes written accross widely varying times
- Documentation spread accross many days
- low hours after work

Low temporal flexibility:
- documentation always done at the same time
- concentrated on very few days each month
- many notes written after hours

In [ ]:
def normalize(df, col):
    return df.groupby("CLINICIAN_ID")[col].transform(
        lambda x: (x - x.min()) / (x.max() - x.min() + 1e-6)
    )

In [ ]:
# --- monthly group ---
monthly = notes.groupby(["CLINICIAN_ID","MONTH"]).agg(
    note_count = ("TEXT","count"),
    avg_tokens = ("WORD_COUNT","mean"),
    hour_std   = ("HOUR_FLOAT","std"),
    after_hours_ratio = ("AFTER_HOURS","mean"),
    active_days = ("DATE","nunique")
).reset_index()

# days in month
monthly["days_in_month"] = monthly["MONTH"].dt.daysinmonth
monthly["active_days_ratio"] = monthly["active_days"] / monthly["days_in_month"]

# handle NaN hour_std (1 note → no std)
monthly["hour_std"] = monthly["hour_std"].fillna(0)

# --- normalization ---
monthly["hour_std_n"] = normalize(monthly, "hour_std")
monthly["after_hours_n"] = normalize(monthly, "after_hours_ratio")
monthly["active_days_n"] = normalize(monthly, "active_days_ratio")

# --- final flexibility score ---
monthly["flexibility_score"] = (
    monthly["hour_std_n"]
    - monthly["after_hours_n"]
    + monthly["active_days_n"]
) / 3

In [ ]:
monthly

In [ ]:
low_flex = monthly.groupby("CLINICIAN_ID")["flexibility_score"].mean().sort_values().head(20)
low_flex

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# ----- choose which column is your flex metric -----
FLEX_COL = "flexibility_score"   # change if needed, e.g. "flex_z"

# average flexibility per clinician
clin_mean_flex = (
    monthly.groupby("CLINICIAN_ID")[FLEX_COL]
           .mean()
           .sort_values()
)

# pick the N lowest-flex clinicians
N_LOW = 10
low_flex_ids = clin_mean_flex.head(N_LOW).index.tolist()
print("Lowest-flex clinicians:", low_flex_ids)

# subset & sort
plot_df = (
    monthly[monthly["CLINICIAN_ID"].isin(low_flex_ids)]
    .sort_values(["CLINICIAN_ID", "MONTH"])
    .copy()
)

In [ ]:
# 3-month rolling average within each clinician
plot_df["flex_smooth"] = (
    plot_df.groupby("CLINICIAN_ID")[FLEX_COL]
           .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
)

In [ ]:
plt.figure(figsize=(18, 10), dpi=150)

for cid, g in plot_df.groupby("CLINICIAN_ID"):
    g = g.sort_values("MONTH")
    # optional: also plot faint raw line underneath for context
    plt.plot(
        g["MONTH"], g[FLEX_COL],
        linestyle="--", linewidth=1, alpha=0.3
    )
    # main smoothed line
    plt.plot(
        g["MONTH"], g["flex_smooth"],
        marker="o", linewidth=2, label=f"Clinician {cid}"
    )

plt.title("Temporal Flexibility Over Time (Lowest-Flex Clinicians)", fontsize=20)
plt.xlabel("Month", fontsize=14)
plt.ylabel("Flexibility (3-month smoothed)", fontsize=14)
plt.xticks(rotation=45)
plt.grid(alpha=0.2)
plt.legend(title="Clinician ID", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

# Build final output table

In [ ]:
# -------------------------------------------------
# BUILD / COMPLETE MONTHLY TEMPORAL FEATURES
# -------------------------------------------------

# 1) Per-clinician-day stats (needed to count "active days")
daily = (
    notes
    .groupby(["CLINICIAN_ID", "DATE"])
    .agg(
        first_time   = ("DATETIME", "min"),
        last_time    = ("DATETIME", "max"),
        day_notes    = ("TEXT", "count")
    )
    .reset_index()
)

# attach MONTH to the daily table
daily["DATE"] = pd.to_datetime(daily["DATE"])
daily["MONTH"] = daily["DATE"].dt.to_period("M").dt.to_timestamp()

# 2) Monthly aggregation
monthly = (
    notes
    .groupby(["CLINICIAN_ID", "MONTH"])
    .agg(
        note_count       = ("TEXT", "count"),
        avg_word_count   = ("WORD_COUNT", "mean"),
        earliest_hour    = ("HOUR_FLOAT", "min"),
        latest_hour      = ("HOUR_FLOAT", "max"),
        hour_mean        = ("HOUR_FLOAT", "mean"),
        hour_std         = ("HOUR_FLOAT", "std"),
        after_hours_ratio= ("AFTER_HOURS", "mean")
    )
    .reset_index()
)

# days_active: distinct days in that month where the clinician wrote ≥1 note
days_active = (
    daily
    .groupby(["CLINICIAN_ID", "MONTH"])["DATE"]
    .nunique()
    .rename("days_active")
    .reset_index()
)

monthly = monthly.merge(days_active,
                        on=["CLINICIAN_ID", "MONTH"],
                        how="left")
monthly["days_active"] = monthly["days_active"].fillna(0)

# -------------------------------------------------
# 3) Add fragmentation proxy: topic_switch_rate per clinician-month
# -------------------------------------------------
def topic_switch_rate(g):
    g = g.sort_values("DATETIME")
    topics = g["TOPIC_ID"].values
    if len(topics) <= 1:
        return 0.0
    switches = (topics[1:] != topics[:-1]).sum()
    return switches / (len(topics) - 1)

frag = (
    notes
    .groupby(["CLINICIAN_ID", "MONTH"])
    .apply(topic_switch_rate)
    .rename("topic_switch_rate")
    .reset_index()
)

monthly = monthly.merge(
    frag,
    on=["CLINICIAN_ID", "MONTH"],
    how="left"
)

monthly["topic_switch_rate"] = monthly["topic_switch_rate"].fillna(0.0)

# -------------------------------------------------
# 4) Proxies for each dimension
# -------------------------------------------------
# Temporal load / scarcity: short notes but many of them
monthly["scarcity_proxy"] = (
    monthly["note_count"] / (monthly["avg_word_count"] + 1e-6)
)

# Temporal flexibility: variability in work hours minus after-hours burden
monthly["flex_proxy"] = monthly["hour_std"].fillna(0.0) - monthly["after_hours_ratio"]

# Temporal fragmentation: use the topic switching rate directly
monthly["fragment_proxy"] = monthly["topic_switch_rate"]

# -------------------------------------------------
# 5) Z-scores per clinician
# -------------------------------------------------
def z_by_clin(df, col):
    return df.groupby("CLINICIAN_ID")[col].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-6)
    )

monthly["scarcity_z"] = z_by_clin(monthly, "scarcity_proxy")
monthly["flex_z"]     = z_by_clin(monthly, "flex_proxy")
monthly["fragment_z"] = z_by_clin(monthly, "fragment_proxy")

print("monthly columns now:")
print(monthly.columns.tolist())

In [ ]:
# -------------------------------------------------
# 0) SANITY CHECK & SORT
# -------------------------------------------------
required_cols = [
    "CLINICIAN_ID", "MONTH",
    "note_count", "avg_word_count",
    "hour_std", "after_hours_ratio", "days_active",
    "scarcity_z", "flex_z", "fragment_z"
]

missing = [c for c in required_cols if c not in monthly.columns]
if missing:
    raise KeyError(f"'monthly' is missing required columns: {missing}")

# Make sure rows are ordered by clinician then time
monthly = monthly.sort_values(["CLINICIAN_ID", "MONTH"]).copy()

# -------------------------------------------------
# 1) HELPER FUNCTIONS FOR TEMPORAL SUMMARIES
# -------------------------------------------------
def last_val(x):
    """Last value in time for a given clinician."""
    return x.iloc[-1]

def last3_mean(x):
    """Mean of the last 3 timepoints (or fewer if <3)."""
    vals = x.tail(3)
    return vals.mean()

def last3_slope(x):
    """
    Simple slope over last 3 timepoints:
    (last - first) / (n-1). If only 1 point, return 0.
    """
    vals = x.tail(3).values
    if len(vals) < 2:
        return 0.0
    return (vals[-1] - vals[0]) / max(len(vals) - 1, 1)

# -------------------------------------------------
# 2) BASE STRUCTURAL TEMPORAL STATS PER CLINICIAN
# -------------------------------------------------
base_temporal = (
    monthly
    .groupby("CLINICIAN_ID")
    .agg(
        months_observed   = ("MONTH", "nunique"),
        mean_note_count   = ("note_count", "mean"),
        total_notes       = ("note_count", "sum"),
        mean_word_count   = ("avg_word_count", "mean"),
        mean_hour_std     = ("hour_std", "mean"),
        mean_after_hours  = ("after_hours_ratio", "mean"),
        mean_days_active  = ("days_active", "mean")
    )
)

# -------------------------------------------------
# 3) LOAD FEATURES (TEMPORAL SCARCITY) PER CLINICIAN
# -------------------------------------------------
LOAD_COL = "scarcity_z"

load_features = (
    monthly
    .groupby("CLINICIAN_ID")[LOAD_COL]
    .agg(
        load_mean   = "mean",
        load_std    = "std",
        load_last   = last_val,
        load_last3  = last3_mean,
        load_slope3 = last3_slope
    )
)

# -------------------------------------------------
# 4) FRAGMENTATION FEATURES (TOPIC SWITCHING) PER CLINICIAN
# -------------------------------------------------
FRAG_COL = "fragment_z"

frag_features = (
    monthly
    .groupby("CLINICIAN_ID")[FRAG_COL]
    .agg(
        frag_mean   = "mean",
        frag_std    = "std",
        frag_last   = last_val,
        frag_last3  = last3_mean,
        frag_slope3 = last3_slope
    )
)

# -------------------------------------------------
# 5) FLEXIBILITY FEATURES PER CLINICIAN
# -------------------------------------------------
FLEX_COL = "flex_z"

flex_features = (
    monthly
    .groupby("CLINICIAN_ID")[FLEX_COL]
    .agg(
        flex_mean   = "mean",
        flex_std    = "std",
        flex_last   = last_val,
        flex_last3  = last3_mean,
        flex_slope3 = last3_slope
    )
)

# -------------------------------------------------
# 6) COMPUTE TII PER CLINICIAN-MONTH
#     (you can adjust weights later if needed)
# -------------------------------------------------

# We need to calculate using the z score because we need to see how far someone is from their own standards
monthly["TII"] = (
    monthly["scarcity_z"]     # more load  → worse
    + monthly["fragment_z"]   # more fragmentation → worse
    - monthly["flex_z"]       # more flexibility   → better, so subtract
) / 3.0

# -------------------------------------------------
# 7) TII FEATURES PER CLINICIAN
# -------------------------------------------------
tii_features = (
    monthly
    .groupby("CLINICIAN_ID")["TII"]
    .agg(
        TII_mean   = "mean",
        TII_std    = "std",
        TII_last   = last_val,
        TII_last3  = last3_mean,
        TII_slope3 = last3_slope
    )
)

# -------------------------------------------------
# 8) JOIN EVERYTHING INTO ONE CLINICIAN-LEVEL TABLE
# -------------------------------------------------
clinician_table = (
    base_temporal
    .join(load_features, how="left")
    .join(frag_features, how="left")
    .join(flex_features, how="left")
    .join(tii_features,  how="left")
)

# Optional: reset index if you want CLINICIAN_ID as a column
clinician_table = clinician_table.reset_index()

print("Final clinician_table shape:", clinician_table.shape)
print(clinician_table.head())

# -------------------------------------------------
# 9) SAVE OUTPUTS (OPTIONAL)
# -------------------------------------------------
monthly.to_csv("temporal_monthly_with_TII.csv", index=False)
# clinician_table.to_csv("temporal_clinician_features_full.csv", index=False)
print("Saved temporal_monthly_with_TII.csv and temporal_clinician_features_full.csv")

In [ ]:
clinician_table.to_csv("temporal_clinician_features_full.csv", index=False)

# Exploring the Output

In [ ]:
DataFrames = {
    'temporal_clinician': pd.read_csv(f"{ROOT}/temp dyn/temporal_clinician_features_full.csv"),
    'temporal_monthly': pd.read_csv(f"{ROOT}/temp dyn/temporal_monthly_with_TII.csv")
}

In [ ]:
temporal_clinician = DataFrames['temporal_clinician']
monthly = DataFrames['temporal_monthly']

In [ ]:
temporal_clinician

## Task
Generate a heatmap to visualize the correlation matrix of numerical features within the `temporal_clinician` DataFrame.

## Correlation Matrix for Clinician Features

### Subtask:
Generate a heatmap to visualize the correlation matrix of numerical features within the `temporal_clinician` DataFrame. This will highlight relationships between various clinician-level aggregated metrics like mean load, fragmentation, and flexibility, as well as TII.


**Reasoning**:
To visualize the correlation between numerical features, I will first select all numerical columns from the `temporal_clinician` DataFrame, calculate their correlation matrix, and then generate a heatmap.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Select only numerical columns
numerical_clinician_features = temporal_clinician.select_dtypes(include=np.number)

# Calculate the correlation matrix
correlation_matrix = numerical_clinician_features.corr()

# Create a heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Clinician Features", fontsize=18)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Correlation Matrix for Monthly Features

### Subtask:
Create a heatmap to display the correlation matrix of numerical features within the `monthly` DataFrame. This will show how monthly metrics such as note count, word count, hour standard deviation, after-hours ratio, topic switch rate, and the Z-scored proxies relate to each other over time.


**Reasoning**:
To visualize the correlations between numerical features in the `monthly` DataFrame, I will first select only the numerical columns, then compute their correlation matrix, and finally generate a heatmap with appropriate labels and styling.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Select only numerical columns from the monthly DataFrame
numerical_monthly_features = monthly.select_dtypes(include=np.number)

# Calculate the correlation matrix
correlation_matrix_monthly = numerical_monthly_features.corr()

# Create a heatmap
plt.figure(figsize=(16, 14))
sns.heatmap(correlation_matrix_monthly, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix of Monthly Features", fontsize=18)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Distribution of Temporal Inflexibility Index (TII)

### Subtask:
Plot a histogram of the 'TII_mean' column from the `temporal_clinician` DataFrame to understand the distribution of the overall Temporal Inflexibility Index across all clinicians. This will help identify the range and common values of TII.


**Reasoning**:
To visualize the distribution of 'TII_mean' as requested, I will generate a histogram using matplotlib and seaborn, setting the specified title and axis labels.



In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(temporal_clinician['TII_mean'].dropna(), bins=30, kde=True)
plt.title('Distribution of Mean Temporal Inflexibility Index (TII)', fontsize=16)
plt.xlabel('Mean TII Score', fontsize=12)
plt.ylabel('Number of Clinicians', fontsize=12)
plt.grid(axis='y', alpha=0.75)
plt.tight_layout()
plt.show()

## TII Trends for Top/Bottom Clinicians

### Subtask:
Generate a time-series line plot showing the 'TII' score over time (by 'MONTH') for clinicians with the highest and lowest average TII scores.


**Reasoning**:
First, I will identify the top 5 clinicians with the highest mean TII scores and the bottom 5 clinicians with the lowest mean TII scores from the `temporal_clinician` DataFrame. Then I will filter the `monthly` DataFrame to include only the data for these selected clinicians.



In [ ]:
top_5_tii_clinicians = temporal_clinician.nlargest(5, 'TII_mean')['CLINICIAN_ID'].tolist()
bottom_5_tii_clinicians = temporal_clinician.nsmallest(5, 'TII_mean')['CLINICIAN_ID'].tolist()

selected_clinicians = top_5_tii_clinicians + bottom_5_tii_clinicians

# Filter the monthly DataFrame for these clinicians
filtered_monthly = monthly[monthly['CLINICIAN_ID'].isin(selected_clinicians)].copy()

# Ensure 'MONTH' is in datetime format for plotting
filtered_monthly['MONTH'] = pd.to_datetime(filtered_monthly['MONTH'])

print("Selected Clinicians for TII Trends:", selected_clinicians)
print("Filtered monthly data for plotting:")
print(filtered_monthly.head())

**Reasoning**:
Now that the relevant clinicians have been identified and the `monthly` data filtered, I will create a time-series line plot to visualize the 'TII' score over time for each of these selected clinicians, ensuring distinct colors and a legend for clarity.



In [ ]:
plt.figure(figsize=(18, 10))

sns.lineplot(data=filtered_monthly, x='MONTH', y='TII', hue='CLINICIAN_ID', marker='o', palette='tab10')

plt.title('TII Trends for Top/Bottom Clinicians Over Time', fontsize=18)
plt.xlabel('Month', fontsize=14)
plt.ylabel('TII Score', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(title='Clinician ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Relationship between TII and its Components

### Subtask:
Create a series of scatter plots to visualize the relationships between the 'TII_mean' and its primary components: 'load_mean', 'frag_mean', and 'flex_mean' from the `temporal_clinician` DataFrame. These plots will help confirm the expected inverse and direct correlations based on the TII calculation.


**Reasoning**:
To visualize the relationships between 'TII_mean' and its components, I will create a series of scatter plots using the `temporal_clinician` DataFrame, arranging them in subplots as specified.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get the min and max for TII_mean to set a common y-axis range
y_min = temporal_clinician['TII_mean'].min()
y_max = temporal_clinician['TII_mean'].max()

plt.figure(figsize=(20, 6))

# Subplot 1: TII vs. Mean Load
plt.subplot(1, 3, 1)
sns.scatterplot(data=temporal_clinician, x='load_mean', y='TII_mean', alpha=0.6)
plt.xlabel('Mean Load (Z-score)', fontsize=12)
plt.ylabel('Mean TII Score', fontsize=12)
plt.title('TII vs. Mean Load', fontsize=14)
plt.ylim(y_min, y_max)
plt.grid(True, linestyle='--', alpha=0.7)

# Subplot 2: TII vs. Mean Fragmentation
plt.subplot(1, 3, 2)
sns.scatterplot(data=temporal_clinician, x='frag_mean', y='TII_mean', alpha=0.6)
plt.xlabel('Mean Fragmentation (Z-score)', fontsize=12)
plt.ylabel('Mean TII Score', fontsize=12)
plt.title('TII vs. Mean Fragmentation', fontsize=14)
plt.ylim(y_min, y_max)
plt.grid(True, linestyle='--', alpha=0.7)

# Subplot 3: TII vs. Mean Flexibility
plt.subplot(1, 3, 3)
sns.scatterplot(data=temporal_clinician, x='flex_mean', y='TII_mean', alpha=0.6)
plt.xlabel('Mean Flexibility (Z-score)', fontsize=12)
plt.ylabel('Mean TII Score', fontsize=12)
plt.title('TII vs. Mean Flexibility', fontsize=14)
plt.ylim(y_min, y_max)
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## Final Task

### Subtask:
Summarize the key insights gleaned from the correlation matrices, TII distribution, and trend plots, highlighting significant relationships and patterns observed in the temporal data.


## Summary:

### Q&A
The analysis provided insights into significant relationships and patterns observed in the temporal data concerning clinician performance and the Temporal Inflexibility Index (TII). Key findings include:

*   **Correlation Matrices**: Heatmaps were generated for both clinician-level and monthly features to visualize their interrelationships.
*   **TII Distribution**: The distribution of mean TII scores across clinicians was plotted to understand its range and common values.
*   **TII Trends**: Time-series plots illustrated TII score trends over time for clinicians with the highest and lowest average TII, revealing temporal patterns.
*   **TII Component Relationships**: Scatter plots confirmed the expected correlations between TII and its constituent components (load, fragmentation, and flexibility).

### Data Analysis Key Findings
*   The analysis confirmed the anticipated relationships between the mean Temporal Inflexibility Index (TII) and its core components: a direct correlation with mean load and mean fragmentation, and an inverse correlation with mean flexibility. This validates the underlying structure of the TII metric.
*   Correlation heatmaps were successfully generated for both clinician-level and monthly aggregated features, providing a visual representation of how various metrics (e.g., mean load, fragmentation, flexibility, note count, word count, after-hours ratio) relate to each other.
*   A histogram of the mean TII scores across clinicians was created, providing a visual understanding of the distribution of TII, which is crucial for identifying typical and extreme inflexibility values.
*   Time-series plots were generated to track the TII scores over time for the top 5 and bottom 5 clinicians based on their average TII. This allowed for the observation of individual clinician TII trends and their stability or fluctuation over monthly periods.

### Insights or Next Steps
*   Investigate specific strong correlations identified in the monthly and clinician feature heatmaps to understand potential causal relationships or underlying operational factors. For example, if "after-hours ratio" strongly correlates with "fragmentation," this could indicate specific challenges or work patterns.
*   Further analyze the temporal patterns observed in the TII trends for top/bottom clinicians to determine if high or low inflexibility is a persistent characteristic or if it changes in response to specific events or interventions.


In [ ]:
monthly = pd.read_csv(paths['monthly'])

In [ ]:
monthly.head()

In [ ]:
active_days = set(monthly['days_active'])

In [ ]:
len([note for note in monthly['days_active'] if note == 11])

In [ ]:
days_active_count = {}
for days in active_days:
  days_active_count[days] = len([note for note in monthly['days_active'] if note == days])

In [ ]:
days_active_count